# 03 - Model Comparison & Thresholds (CPU)
Compare feature variants (raw vs clipped outliers vs log amount vs balance deltas) across LR/RF/XGB; export thresholds and summaries to artifacts/model_eval.

## 1) Environment Setup
CPU-only; no Colab. Uses pandas/numpy/sklearn/matplotlib; optional XGBoost if installed. Artifacts → artifacts/model_eval.

In [ ]:
import os
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_recall_curve, roc_curve, auc
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

plt.style.use("dark_background")
sns.set_theme(style="darkgrid")

ROOT = Path(".").resolve()
DATA_PATH = Path(os.getenv("FRAUD_DATA_PATH", ROOT / "data" / "processed" / "transactions.parquet"))
ARTIFACT_DIR = ROOT / "artifacts" / "model_eval"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RNG_SEED = 42
np.random.seed(RNG_SEED)

print(f"Using data path: {DATA_PATH}")
print(f"Artifacts → {ARTIFACT_DIR}")

## 2) Load Dataset + Base Feature Engineering
Synthesize if data missing (~1% fraud). Adds mandatory features: amount_log, balance deltas, outlier flags.

In [ ]:
def synthesize_transactions(n_rows: int = 50000, fraud_ratio: float = 0.01, rng_seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(rng_seed)
    labels = rng.choice([0, 1], size=n_rows, p=[1 - fraud_ratio, fraud_ratio])
    amount = rng.gamma(shape=2.0, scale=200.0, size=n_rows)
    oldbalanceOrg = rng.normal(loc=5000, scale=1500, size=n_rows)
    newbalanceOrig = oldbalanceOrg - amount * rng.uniform(0.8, 1.0, size=n_rows)
    oldbalanceDest = rng.normal(loc=2000, scale=1000, size=n_rows)
    newbalanceDest = oldbalanceDest + amount * rng.uniform(0.7, 1.0, size=n_rows)
    tx_type = rng.choice(["PAYMENT", "TRANSFER", "CASH_OUT", "DEBIT"], size=n_rows)
    return pd.DataFrame({
        "amount": amount,
        "oldbalanceOrg": oldbalanceOrg,
        "newbalanceOrig": newbalanceOrig,
        "oldbalanceDest": oldbalanceDest,
        "newbalanceDest": newbalanceDest,
        "type": tx_type,
        "is_fraud": labels,
    })


def load_dataset(path: Path) -> pd.DataFrame:
    if path.exists():
        if path.suffix.lower() == ".parquet":
            df = pd.read_parquet(path)
        else:
            df = pd.read_csv(path)
        print(f"Loaded dataset from {path} with shape {df.shape}")
        return df
    print(f"Data path not found: {path}. Synthesizing sample dataset (~1% fraud).")
    return synthesize_transactions()


def add_base_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["amount_log"] = np.log1p(df["amount"].clip(lower=0))
    df["balance_delta_org"] = df["oldbalanceOrg"] - df["newbalanceOrig"]
    df["balance_delta_dest"] = df["newbalanceDest"] - df["oldbalanceDest"]
    for col in ["amount", "balance_delta_org", "balance_delta_dest"]:
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        df[f"outlier_flag_{col}"] = ((df[col] < lower) | (df[col] > upper)).astype(int)
    return df


df_raw = load_dataset(DATA_PATH)
df_base = add_base_features(df_raw)

## 3) Feature Variants
- Variant A: raw engineered (base)
- Variant B: clipped outliers
- Variant C: log-scaled amount emphasized (amount_log only)
- Variant D: balance delta focused
Evaluate each across models.

In [ ]:
def clip_outliers(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    df = df.copy()
    for col in cols:
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        df[col] = df[col].clip(lower=lower, upper=upper)
    return df


variants = {}
variants["A_base"] = df_base
variants["B_clipped"] = clip_outliers(df_base, ["amount", "balance_delta_org", "balance_delta_dest"])
variants["C_amount_log_only"] = df_base.assign(amount=df_base["amount_log"]).drop(columns=["amount_log"], errors="ignore")
variants["D_balance_delta_focus"] = df_base[[
    "balance_delta_org", "balance_delta_dest", "amount", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest", "type", "is_fraud"
]]

list(variants.keys())

## 4) Train/Test Split per Variant
Use consistent stratified split across variants for fairness.

In [ ]:
splits = {}

for name, dfv in variants.items():
    X = dfv.drop(columns=["is_fraud"])
    y = dfv["is_fraud"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RNG_SEED, stratify=y
    )
    splits[name] = (X_train, X_test, y_train, y_test)
    print(f"Variant {name}: train {X_train.shape}, test {X_test.shape}, fraud ratio {y_train.mean():.4f}")

## 5) Models and Evaluation Helpers
Shared model configs and metric computation (PR AUC, ROC AUC, precision/recall/f1).

In [ ]:
def build_models(preprocessor):
    models = {}
    models["lr"] = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("clf", LogisticRegression(max_iter=400, class_weight="balanced", n_jobs=-1, C=0.5)),
    ])
    models["rf"] = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("clf", RandomForestClassifier(
            n_estimators=160,
            max_depth=14,
            min_samples_leaf=2,
            n_jobs=-1,
            class_weight="balanced_subsample",
            random_state=RNG_SEED,
        )),
    ])
    if HAS_XGB:
        models["xgb"] = Pipeline(steps=[
            ("preprocess", preprocessor),
            ("clf", XGBClassifier(
                max_depth=6,
                n_estimators=220,
                learning_rate=0.07,
                subsample=0.9,
                colsample_bytree=0.9,
                reg_lambda=1.1,
                tree_method="hist",
                objective="binary:logistic",
                eval_metric="aucpr",
                random_state=RNG_SEED,
            )),
        ])
    return models


def evaluate_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_test)[:, 1]
    pr, rc, _ = precision_recall_curve(y_test, probs)
    fpr_curve, tpr_curve, _ = roc_curve(y_test, probs)
    pr_auc = auc(rc, pr)
    roc_auc = auc(fpr_curve, tpr_curve)
    precision_at_50 = (probs >= 0.5).astype(int)
    tp = ((precision_at_50 == 1) & (y_test == 1)).sum()
    fp = ((precision_at_50 == 1) & (y_test == 0)).sum()
    fn = ((precision_at_50 == 0) & (y_test == 1)).sum()
    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    return {
        "pr_auc": pr_auc,
        "roc_auc": roc_auc,
        "precision@0.5": precision,
        "recall@0.5": recall,
        "probs": probs,
        "pr_curve": (pr, rc),
        "roc_curve": (fpr_curve, tpr_curve),
    }

## 6) Run Comparisons Across Variants
Train/eval LR/RF/(XGB) on each variant; collect metrics and best thresholds by Youden J on ROC + PR best-F1.

In [ ]:
def best_thresholds(y_true, probs):
    pr, rc, th_pr = precision_recall_curve(y_true, probs)
    fpr, tpr, th_roc = roc_curve(y_true, probs)
    f1s = 2 * (pr * rc) / (pr + rc + 1e-9)
    best_f1_idx = np.argmax(f1s)
    best_f1 = f1s[best_f1_idx]
    best_pr_threshold = th_pr[best_f1_idx - 1] if best_f1_idx > 0 else 0.5
    youden = tpr - fpr
    best_roc_idx = np.argmax(youden)
    best_roc_threshold = th_roc[best_roc_idx]
    return best_pr_threshold, best_f1, best_roc_threshold


summary_rows = []

for vname, (X_train, X_test, y_train, y_test) in splits.items():
    cat_cols = [c for c in X_train.columns if X_train[c].dtype == object]
    num_cols = [c for c in X_train.columns if c not in cat_cols]
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("scaler", StandardScaler())]), num_cols),
            ("cat", Pipeline([("encoder", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
        ]
    )

    models = build_models(preprocessor)
    for mname, model in models.items():
        res = evaluate_model(model, X_train, y_train, X_test, y_test)
        best_pr_th, best_f1, best_roc_th = best_thresholds(y_test.to_numpy(), res["probs"])
        summary_rows.append({
            "variant": vname,
            "model": mname,
            "pr_auc": res["pr_auc"],
            "roc_auc": res["roc_auc"],
            "precision@0.5": res["precision@0.5"],
            "recall@0.5": res["recall@0.5"],
            "best_pr_threshold": best_pr_th,
            "best_f1": best_f1,
            "best_roc_threshold": best_roc_th,
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.sort_values(by="pr_auc", ascending=False).head()

## 7) Plot Top Models and Save Thresholds
Pick top 3 by PR AUC; plot PR/ROC; save thresholds.json and summary CSV.

In [ ]:
top3 = summary_df.sort_values(by="pr_auc", ascending=False).head(3)

fig_pr, ax_pr = plt.subplots(figsize=(8, 6))
fig_roc, ax_roc = plt.subplots(figsize=(8, 6))

# Recompute curves for top3 to plot
for _, row in top3.iterrows():
    vname = row["variant"]
    mname = row["model"]
    X_train, X_test, y_train, y_test = splits[vname]
    cat_cols = [c for c in X_train.columns if X_train[c].dtype == object]
    num_cols = [c for c in X_train.columns if c not in cat_cols]
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("scaler", StandardScaler())]), num_cols),
            ("cat", Pipeline([("encoder", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
        ]
    )
    model = build_models(preprocessor)[mname]
    res = evaluate_model(model, X_train, y_train, X_test, y_test)
    pr, rc = res["pr_curve"]
    fpr_curve, tpr_curve = res["roc_curve"]
    ax_pr.plot(rc, pr, label=f"{vname}-{mname} (AUC={res['pr_auc']:.3f})")
    ax_roc.plot(fpr_curve, tpr_curve, label=f"{vname}-{mname} (AUC={res['roc_auc']:.3f})")

ax_pr.set_xlabel("Recall")
ax_pr.set_ylabel("Precision")
ax_pr.set_title("Top3 Precision-Recall")
ax_pr.legend()

ax_roc.set_xlabel("FPR")
ax_roc.set_ylabel("TPR")
ax_roc.set_title("Top3 ROC")
ax_roc.legend()

pr_path = ARTIFACT_DIR / "top3_pr_curves.png"
roc_path = ARTIFACT_DIR / "top3_roc_curves.png"
fig_pr.savefig(pr_path)
fig_roc.savefig(roc_path)
plt.close(fig_pr)
plt.close(fig_roc)
print(f"Saved top3 PR/ROC curves to {pr_path}, {roc_path}")

summary_path = ARTIFACT_DIR / "model_variant_comparison.csv"
summary_df.to_csv(summary_path, index=False)
print(f"Saved summary to {summary_path}")

thresholds_json = {
    f"{r['variant']}_{r['model']}": {
        "best_pr_threshold": float(r["best_pr_threshold"]),
        "best_roc_threshold": float(r["best_roc_threshold"]),
        "pr_auc": float(r["pr_auc"]),
        "roc_auc": float(r["roc_auc"]),
    }
    for _, r in summary_df.iterrows()
}

import json
th_path = ARTIFACT_DIR / "thresholds.json"
th_path.write_text(json.dumps(thresholds_json, indent=2))
print(f"Saved thresholds to {th_path}")

summary_df.sort_values(by="pr_auc", ascending=False).head(10)